In [2]:
import logging
from minio import Minio
from minio.error import S3Error
import os

from s3_torch_data_loader import get_minio_credentials, get_object_list, get_object_from_minio

DATASET_ROOT_FOLDER = '/home/keithpij/dlio_benchmark/data/unet3d' 
BUCKET_NAME = os.environ['BUCKET_NAME']

In [3]:
def put_folder(bucket_name: str, dataset_root_folder: str, split: str) -> int:
    '''
    Removes all objects from the specified bucket.
    '''
    url, access_key, secret_key, secure = get_minio_credentials()

    try:
        # Create client with access and secret key
        client = Minio(url,  # host.docker.internal
                    access_key,  
                    secret_key, 
                    secure=secure)

        # Make the bucket if it does not exist.
        found = client.bucket_exists(bucket_name)
        if not found:
            client.make_bucket(bucket_name)
            logging.info(f'Creating {bucket_name}.')
        else:
            logging.info(f'Bucket {bucket_name} already exists.')

        count = 0
        samples_dir = os.path.join(dataset_root_folder, split)
        for entry in os.listdir(samples_dir):
            sample_file_path = os.path.join(samples_dir, entry)
            sample_object_path = f'{split}/{entry}'
            client.fput_object(bucket_name, sample_object_path, sample_file_path)
            count += 1
            if count % 500 == 0:
                logger.info(f'{count} objects uploaded to {bucket_name}.')

    except S3Error as s3_err:
        logging.error(f'S3 Error occurred: {s3_err}.')
        raise s3_err
    except Exception as err:
        logging.error(f'Error occurred: {err}.')
        raise err

    return count

In [4]:
count = 0
split = 'train'
samples_dir = os.path.join(DATASET_ROOT_FOLDER, split)
for entry in os.listdir(samples_dir):
    sample_file_path = os.path.join(samples_dir, entry)
    sample_object_path = f'{split}/{entry}'
    print(sample_file_path, sample_object_path)
    count += 1
    if count > 10: break

/home/keithpij/dlio_benchmark/data/unet3d/train/img_45_of_50.npz train/img_45_of_50.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_12_of_50.npz train/img_12_of_50.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_28_of_50.npz train/img_28_of_50.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_29_of_50.npz train/img_29_of_50.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_08_of_50.npz train/img_08_of_50.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_01_of_50.npz train/img_01_of_50.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_24_of_50.npz train/img_24_of_50.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_23_of_50.npz train/img_23_of_50.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_10_of_50.npz train/img_10_of_50.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_32_of_50.npz train/img_32_of_50.npz
/home/keithpij/dlio_benchmark/data/unet3d/train/img_15_of_50.npz train/img_15_of_50.npz


In [5]:
put_folder(BUCKET_NAME, DATASET_ROOT_FOLDER, 'train')

50

In [6]:
object_list = get_object_list(BUCKET_NAME, prefix='train/')
len(object_list)
print(object_list[:10])

['train/img_00_of_50.npz', 'train/img_01_of_50.npz', 'train/img_02_of_50.npz', 'train/img_03_of_50.npz', 'train/img_04_of_50.npz', 'train/img_05_of_50.npz', 'train/img_06_of_50.npz', 'train/img_07_of_50.npz', 'train/img_08_of_50.npz', 'train/img_09_of_50.npz']
